Récupération des données

On obtient la cle api de météo france à l'aide de getpass (interactif avec celui qui lance la commande).
Le .strip() est fait pour supprimer les espace et les sauts de lignes du mot (ça évite des erreurs lors du copié collé)

!!!il faudrait donner le lien du site ou obtenir une clé!!!

Choisir les stations avec liste-stations/quotidienne.
Commander les données d’une station avec commande-station/quotidienne, qui renvoie un numéro de commande.
Récupérer le fichier avec commande/fichier en donnant ce numéro.

In [2]:
import requests 
import time
from getpass import getpass
import pandas as pd
import io

url_stations = 'https://public-api.meteofrance.fr/public/DPClim/v1/liste-stations/quotidienne?id-departement=95&parametre=vent'

CLE_API = getpass("Clé API Météo-France : ").strip()

'''
Récupération de l'id des stations d'intérêt
'''
id_stations = []

r_stations = requests.get(url_stations, headers={"apikey": CLE_API})
liste_stations = r_stations.json()
id_stations += [liste_stations[i]['id'] for i in range(len(liste_stations))]
print(id_stations)

['95078001', '95088001', '95306001', '95527001', '95580001', '95678001', '95690001']


In [3]:
'''
Récupération du code commande associé à chaque station
'''

codes_commande = []
liste_annee = [2025]
for annee in liste_annee:
    for station in id_stations:
        
        url_code_commande = f'https://public-api.meteofrance.fr/public/DPClim/v1/commande-station/quotidienne?id-station={station}&date-deb-periode={annee}-01-01T00%3A00%3A00Z&date-fin-periode={annee}-12-31T23%3A59%3A59Z'
        r_code_commande = requests.get(url_code_commande, headers = {"apikey": CLE_API})
        print(r_code_commande)
        codes_commande.append(r_code_commande.json()['elaboreProduitAvecDemandeResponse']['return'])

<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>


Certaines stations sont actuellement fermées donc pour certaines périodes il se peut qu'il n'y ait pas de données. Bien que l'API commande nous donne bien un code pour télécharger le fichier, lorsque l'on essaye l'API téléchargement avec, ça nous renvoie un statut 404 (pas de ressources trouvées), ce qui fait planter le code. Pour contourner cela, on vérifie à chaque fois que le statut est bien 201 (requête réussite).

In [4]:
'''
Récupération des fichiers des données des commandes
'''

time.sleep(3)  #on attend 3 secondes que les fichiers soient créés
liste_df = []
variables_vent = ['POSTE', 'DATE', 'FFM', 'FF2M', 'FXY', 'DXY', 'FXI','DXI', 'FXI2', 'DXI2', 'FXI3S', 'DXI3S']
for code in codes_commande:

    url_telechargement_donnees = f'https://public-api.meteofrance.fr/public/DPClim/v1/commande/fichier?id-cmde={code}'
    r_telechargement_donnees = requests.get(url_telechargement_donnees, headers = {'apikey': CLE_API})
    print(r_telechargement_donnees)
    if r_telechargement_donnees.status_code == 201:
        df_donnees = pd.read_csv(io.StringIO(r_telechargement_donnees.text), sep=";", decimal=",",usecols=variables_vent, dtype={"POSTE": str}, parse_dates=["DATE"], date_format="%Y%m%d")  # garde les zéros initiaux de l'identifiant (POSTE)
        liste_df.append(df_donnees)

df_donnees_entier = pd.concat(liste_df, ignore_index=True)
df_donnees_entier

<Response [201]>
<Response [201]>
<Response [404]>
<Response [201]>
<Response [201]>
<Response [404]>
<Response [201]>


,POSTE,DATE,FFM,FXI,DXI,FXY,DXY,FF2M,FXI2,DXI2,FXI3S,DXI3S
0,95078001,2025-01-01,8.2,18.3,220.0,11.6,220.0,NaN,NaN,NaN,17.2,210.0
1,95078001,2025-01-02,2.7,7.5,360.0,6.1,280.0,NaN,NaN,NaN,7.2,360.0
2,95078001,2025-01-03,1.4,5.5,350.0,3.7,330.0,NaN,NaN,NaN,5.2,350.0
3,95078001,2025-01-04,3.4,8.6,120.0,5.7,120.0,NaN,NaN,NaN,8.3,120.0
4,95078001,2025-01-05,7.5,18.5,220.0,10.6,210.0,NaN,NaN,NaN,16.2,220.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1820,95690001,2025-12-27,3.7,9.4,80.0,5.9,80.0,NaN,NaN,NaN,8.6,80.0
1821,95690001,2025-12-28,3.7,10.5,70.0,6.3,50.0,NaN,NaN,NaN,9.1,40.0
1822,95690001,2025-12-29,3.3,10.0,60.0,6.3,40.0,NaN,NaN,NaN,9.6,60.0
1823,95690001,2025-12-30,3.4,8.4,40.0,5.1,40.0,NaN,NaN,NaN,7.6,40.0


## Limites de l'API : pourquoi on ne collecte pas tout par requêtes

L'API de Météo-France est limitée à 100 requêtes par minute. Or, pour obtenir les données d'une station sur une période, il faut :

1. une requête de commande (commande-station/quotidienne) ;
2. une requête de téléchargement (commande/fichier)

Soit au minimum 2 requêtes par station et par année (période maximale d'une seule requête). Sachant qu'il y a 96 départements et qu'il y a une moyenne d'environ 24 (ou 40?) stations par département, cela fait beaucoup trop de reqêtes pour importer toutes les données.


### Conclusion

Cette API convient pour quelques stations ou une période courte, mais elle est loin d'être suffisante pour collecter l'ensemble des départements sur une longue période. Pour ce besoin, il faut passer par les fichiers départementaux en téléchargement direct (jeu « Données climatologiques de base -quotidiennes » sur data.gouv.fr).

Les liens de téléchargement des fichiers de données sont récupérés via l'API de data.gouv.fr (GET /api/1/datasets/{dataset}/), dont la réponse JSON contient, dans le champ resources, la liste des fichiers du jeu « Données climatologiques de base - quotidiennes », chacun avec son titre, son format et son lien permanent (latest).

## Récupération des données de vent depuis data.gouv.fr

Ce code récupère automatiquement les données de vent de tous les départements, sans télécharger les fichiers à la main :

1. **Liste des fichiers** : une requête à l'API de data.gouv.fr (`/api/1/datasets/...`) renvoie la fiche du jeu « Données climatologiques de base - quotidiennes ». L'en-tête `X-Fields` limite la réponse aux champs utiles (titre, format, type et lien permanent `latest` de chaque fichier).
2. **Sélection des fichiers d'intérêt** : une expression régulière (`QUOT_departement_\d+_periode_1950-2024_RR-T-Vent`) ne garde que les fichiers départementaux couvrant la période 1950-2024 et contenant les variables de précipitations, de température et de vent. On obtient une liste de dictionnaires, avec un fichier par département (DROM et COM inclus).
3. **Lecture de chaque fichier** : chaque fichier compressé (`csv.gz`) est lu directement depuis son lien, en ne gardant que quatre colonnes : `NUM_POSTE` (station, lue comme texte pour conserver les zéros initiaux), `AAAAMMJJ` (date), `FFM` (vent moyen) et `FXI3S` (rafale maximale).
4. **Filtre temporel** : seules les lignes à partir du 1er janvier 2016 sont conservées.
5. **Fusion** : les tableaux de tous les départements sont concaténés en un seul DataFrame, `df_final`.

**Temps d'exécution** : Les fichiers sont lus depuis le réseau à chaque exécution, ce qui est lent. (environ 5 minutes)

In [5]:
import re

r = requests.get(
    "https://www.data.gouv.fr/api/1/datasets/donnees-climatologiques-de-base-quotidiennes/",
    headers={"X-Fields": "id,title,resources{title,format,type,latest}"},
    timeout=30,
)

fiche = r.json()
# for i in range(len(fiche['resources'])):
#     print(fiche['resources'][i]['title'])

pattern = r'QUOT_departement_\d+_periode_1950-2024_RR-T-Vent'

fichiers_interet = [fiche['resources'][i] for i in range(len(fiche['resources'])) if re.fullmatch(pattern, fiche['resources'][i]['title']) ] ## On filtre sur les jeux de données qui nous intéressent
##liste de dictionnaires
##inclu les DROM et les COM

# print(fichiers_interet[0]['latest'])

variables_vent_1 = ['NUM_POSTE', 'AAAAMMJJ', 'FFM', 'FXI3S'] ##, 'FF2M', 'FXY', 'DXY', 'FXI','DXI', 'FXI2', 'DXI2', 'FXI3S', 'DXI3S'
donnees_vent = []
nbre_departement = 0
for i in range(len(fichiers_interet)): ##len(fichiers_interet)
    print(f"nombre de départements importés : {nbre_departement}", end="\r")
    donnees_interet = pd.read_csv(fichiers_interet[i]['latest'],sep=';',dtype = {'NUM_POSTE' : str}, usecols = variables_vent_1, compression="gzip")  ##, parse_dates=["AAAAMMJJ"], date_format="%Y%m%d
    donnes_interet_filtre = donnees_interet[donnees_interet["AAAAMMJJ"] >= 20160101]
    donnees_vent.append(donnes_interet_filtre)
    nbre_departement += 1
print(f"nombre de départements importés : {len(fichiers_interet)}")

df_final = pd.concat(donnees_vent, ignore_index=True)

df_final.info()
display(df_final)

nombre de départements importés : 106
<class 'pandas.DataFrame'>
RangeIndex: 10273735 entries, 0 to 10273734
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   NUM_POSTE  str    
 1   AAAAMMJJ   int64  
 2   FFM        float64
 3   FXI3S      float64
dtypes: float64(2), int64(1), str(1)
memory usage: 393.1 MB


,NUM_POSTE,AAAAMMJJ,FFM,FXI3S
0,01014002,20160101,2.0,14.2
1,01014002,20160102,1.1,10.2
2,01014002,20160103,2.6,17.4
3,01014002,20160104,2.9,12.3
4,01014002,20160105,1.8,7.4
...,...,...,...,...
10273730,99138001,20170727,NaN,NaN
10273731,99138001,20170728,NaN,NaN
10273732,99138001,20170729,NaN,NaN
10273733,99138001,20170730,NaN,NaN


## Préparation du tableau de données

À partir du tableau d'origine (une ligne par station et par jour), on a effectué trois transformations :

1. **Création de la variable `departement`** : elle est déduite des deux (ou trois) premiers chiffres du numéro de station (`NUM_POSTE`), qui correspondent au code du département.
2. **Agrégation par département** : pour chaque département (et chaque année), on a calculé la **moyenne** des valeurs de l'ensemble des stations du département, afin d'obtenir une seule mesure par département. Les valeurs manquantes (`NaN`) sont ignorées par le calcul de la moyenne, qui porte donc sur les valeurs disponibles. Cela évite de les traiter comme des zéros. Cependant une moyenne calculée sur peu de mesures est moins fiable.
3. **Renommage des variables** pour les rendre plus lisibles :
   - `FFM` devient `vent_moyen_ms` : moyenne quotidienne du vent moyenné sur 10 minutes, à 10 m (en m/s) ;
   - `FXI3S` devient `rafale_max_ms` : maximum quotidien du vent moyenné sur 3 secondes, à 10 m (en m/s).

**Limite** : la moyenne départementale mélange des stations de localisation et d'altitude différentes, et le nombre de stations peut varier selon les départements et les années.

In [ ]:

df_final_int = df_final.rename(columns = {'AAAAMMJJ' : 'DATE'})
df_final_int['DATE'] = pd.to_datetime(df_final_int['DATE'], format = '%Y%m%d' )
df_final1 = df_final_int.rename(columns={"FFM": "vent_moyen_ms","FXI3S": "rafale_max_ms"})


df_final1['dep'] = df_final1['NUM_POSTE'].str[:2]
df_final1.loc[df_final1['dep'].astype(int) > 95, "dep"] = df_final1.loc[df_final1['dep'].astype(int) > 95, "NUM_POSTE"].str[:3]
df_final1["semaine_iso"] = df_final1["DATE"].dt.isocalendar().week
df_final1["annee_iso"] = df_final1["DATE"].dt.isocalendar().year
df_final1["cle_semaine"] = list(zip(df_final1["dep"], df_final1["semaine_iso"], df_final1["annee_iso"]))
df_final1 = df_final1[["DATE", "annee_iso", "semaine_iso", "dep", "cle_semaine", "vent_moyen_ms", "rafale_max_ms"]]

display(df_final1)




df_group_DATE = df_final1.groupby(['DATE', 'dep'])[['vent_moyen_ms', 'rafale_max_ms']].mean().reset_index()
df_group_semaine = df_final1.groupby(['cle_semaine'])[['vent_moyen_ms', 'rafale_max_ms']].mean().reset_index()

# print(df_group_DATE.describe())
display(df_group_semaine)
print(df_group_semaine.info())
display(df_group_DATE)
print(df_group_DATE.info())





visualisation map géo

In [ ]:
pd.set_option('display.min_rows', 200)   # nb de lignes affichées en mode tronqué
pd.set_option('display.min_columns', 200)

df_visu = df_final1[df_final1["dep"].astype(int) <= 95]  
df_visu_test = df_visu[['dep']].groupby('dep').size().reset_index(name = "nombre_occurence")
display(df_group_semaine)

OptionError: No such keys(s): 'display.min_columns'

In [57]:
donnee_vent_moyen_ms = df_final1[df_final1["dep"].astype(int)<=95].groupby(['dep'])[['vent_moyen_ms']].mean().reset_index()  ### ici il faut choisir les dep que je veux inclure

###on créer la corse 2A 2B pour que ça match avec l'autre tableau
corse = {"dep": ["2A", "2B"], "vent_moyen_ms": [donnee_vent_moyen_ms.loc[donnee_vent_moyen_ms["dep"] == "20"], donnee_vent_moyen_ms.loc[donnee_vent_moyen_ms["dep"]  == "20"]]}
corse_df = pd.DataFrame(corse)
display(corse_df)
donnee_vent_moyen_ms[donnee_vent_moyen_ms["dep"] == "20"]

# display(donnee_vent_moyen_ms)
# print(donnee_vent_moyen_ms.describe())




,dep,vent_moyen_ms
0,2A,dep vent_moyen_ms 19 20 3.594595
1,2B,dep vent_moyen_ms 19 20 3.594595


,dep,vent_moyen_ms
19,20,3.594595


In [43]:
!pip install py7zr geopandas openpyxl tqdm s3fs ipywidgets --quiet
!pip install PyYAML --quiet
!pip install cartiflette --quiet

import geopandas as gpd
import matplotlib.pyplot as plt
from cartiflette import carti_download

departements = carti_download(
    values="France",
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="geojson",
    filter_by="FRANCE_ENTIERE",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

# departements_metropole = departements[departements["INSEE_DEP"].astype(int) <= 95]

display(departements)


# carte = departements.merge(
#     donnee_vent_moyen_ms,
#     left_on="INSEE_DEP",
#     right_on="dep",
#     how="left"
# )
# # display(carte)
# carte = carte.to_crs(2154)
# # display(carte)

# carte.plot(
#     column="vent_moyen_ms",
#     cmap="RdYlBu_r",
#     legend=True,
#     edgecolor="black",
#     linewidth=0.5,
#     figsize=(12, 10),
#     missing_kwds={"color": "lightgrey"}
# )

# plt.title("Vitesse moyenne du vent par département")
# plt.axis("off")
# plt.show()


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


,INSEE_DEP,PAYS,LIBELLE_DEPARTEMENT,POPULATION,SOURCE,geometry
0,971,France,Guadeloupe,384239,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-61.62648 16.27275, -61.62623 ..."
1,973,France,Guyane,281678,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-51.6408 4.21282, -51.64144 4...."
2,972,France,Martinique,364508,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-60.93246 14.7306, -60.9326 14..."
3,976,France,Mayotte,256518,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((45.08357 -12.96139, 45.08547 -..."
4,43,France,Haute-Loire,227570,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((3.89745 45.35699, 3.89691 45.35631, ..."
5,65,France,Hautes-Pyrénées,229567,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-0.09705 43.5822, -0.09642 43...."
6,29,France,Finistère,915090,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-3.97907 47.70396, -3.98004 47..."
7,22,France,Côtes-d'Armor,600582,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-3.20174 48.85198, -3.20193 48..."
8,38,France,Isère,1271166,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((5.10149 45.81356, 5.09828 45.80627, ..."
9,86,France,Vienne,438435,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((-0.1021 47.06517, -0.10128 47.06247,..."
